# DSTS — LLM Inference Optimisation: Full Benchmark Suite

This notebook clones the repo, runs all experiments on GPU, and zips `results/` for download.

## Prerequisites (do these ONCE before running)

### 1. HuggingFace token (for Llama-3.2-1B-Instruct access)
1. Go to https://huggingface.co/settings/tokens → create a token with **Read** scope
2. Accept the Llama-3.2 license at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
3. In Kaggle: top-right avatar → **Settings** → **Secrets** → **Add a new secret**
   - Name: `HF_TOKEN`
   - Value: your HuggingFace token (starts with `hf_...`)
4. In this notebook: click the **🔑 Add-ons** menu → **Secrets** → toggle `HF_TOKEN` to **Attached**

### 2. Kaggle Dataset with pretrained artefacts (skip ~40 min of training)
Upload `router_checkpoint.pt` (7.5 MB) and `mlp_transition_graph.pt` (31 MB) as a **private Kaggle Dataset**:
1. Kaggle → **Datasets** → **New Dataset**
2. Name it exactly: **`dsts-artefacts`** (slug will be `<your-username>/dsts-artefacts`)
3. Upload the two `.pt` files from your local `results/` directory
4. Click **Create** → wait for upload to finish
5. In this notebook: click **+ Add Data** (right panel) → search for `dsts-artefacts` → **Add**

### 3. GPU runtime
- Notebook Settings → Accelerator → **GPU T4 x2** or **P100** (P100 preferred for single-GPU)

---
**After running:** go to the notebook's **Output** tab → download `results.zip`  
**Locally:** unzip into your `dsts/results/` folder, then run `python run_plots.py`

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NOT FOUND — check accelerator settings')

import torch
print(f'torch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')
    print(f'CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: HuggingFace token ─────────────────────────────────────────────────
# Reads HF_TOKEN from Kaggle Secrets and sets it in the environment.
# See Prerequisites above for how to add the secret.
import os
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token  # legacy env var
    print('HF_TOKEN loaded from Kaggle Secrets ✓')
except Exception as e:
    print(f'WARNING: Could not load HF_TOKEN from Kaggle Secrets: {e}')
    print('You can set it manually:')
    print('  os.environ["HF_TOKEN"] = "hf_...your_token..."')
    raise

In [ ]:
# ── Cell 3: Install dependencies ──────────────────────────────────────────────
# Kaggle already has torch/transformers; we pin versions and add missing packages.
!pip install -q \
    'transformers>=4.40.0' \
    'accelerate>=0.29.0' \
    'datasets>=2.18.0' \
    'sentencepiece>=0.2.0' \
    'tokenizers>=0.19.0' \
    'faiss-cpu' \
    'scikit-learn>=1.4.0' \
    'sentence-transformers>=2.7.0' \
    'tqdm>=4.66.0' \
    'numpy>=1.26.0' \
    'pandas>=2.2.0' \
    'matplotlib>=3.8.0' \
    'seaborn>=0.13.0'

print('Dependencies installed ✓')

In [ ]:
# ── Cell 4: Clone repo ────────────────────────────────────────────────────────
import os

WORKDIR = '/kaggle/working/dsts'

if not os.path.exists(WORKDIR):
    !git clone https://github.com/deil87/dsts.git {WORKDIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {WORKDIR} pull

os.chdir(WORKDIR)
print(f'Working directory: {os.getcwd()}')
!ls -la

In [ ]:
# ── Cell 5: Copy pretrained artefacts from Kaggle Dataset ────────────────────
# These skip ~40 min of training (router training + MLP graph construction).
# The dataset must be attached — see Prerequisites above.
#
# IMPORTANT: replace <your-kaggle-username> with your actual Kaggle username,
# or just check what path Kaggle mounted the dataset at with:
#   !ls /kaggle/input/

import shutil, os

os.makedirs('results', exist_ok=True)

# Auto-detect dataset path (works regardless of username)
DATASET_ROOT = None
for candidate in os.listdir('/kaggle/input'):
    candidate_path = f'/kaggle/input/{candidate}'
    if os.path.isfile(f'{candidate_path}/router_checkpoint.pt'):
        DATASET_ROOT = candidate_path
        break

if DATASET_ROOT:
    shutil.copy(f'{DATASET_ROOT}/router_checkpoint.pt', 'results/router_checkpoint.pt')
    shutil.copy(f'{DATASET_ROOT}/mlp_transition_graph.pt', 'results/mlp_transition_graph.pt')
    print(f'Artefacts copied from {DATASET_ROOT} ✓')
    print('  router_checkpoint.pt  — DE router (skips ~30 min training)')
    print('  mlp_transition_graph.pt — MLP graph (skips ~10 min build)')
else:
    print('WARNING: dsts-artefacts dataset not found in /kaggle/input/')
    print('Available datasets:', os.listdir('/kaggle/input'))
    print()
    print('The experiments will still run but will rebuild these artefacts from scratch.')
    print('This adds ~40 minutes to the total runtime.')

In [ ]:
# ── Cell 6: Run baseline ──────────────────────────────────────────────────────
# Measures: tok/s, lm_head fraction, baseline PPL
# Expected runtime: ~5 min
print('=' * 60)
print('EXPERIMENT 1/9: Baseline')
print('=' * 60)
!python run_baseline.py

In [ ]:
# ── Cell 7: Static + Cosine router ───────────────────────────────────────────
# Vocabulary pruning with static weight-norm ranking and cosine similarity.
# Expected runtime: ~15 min
print('=' * 60)
print('EXPERIMENT 2/9: Static + Cosine Router')
print('=' * 60)
!python run_static.py

In [ ]:
# ── Cell 8: Cluster router ────────────────────────────────────────────────────
# k-means cluster-based vocabulary pruning.
# Expected runtime: ~10 min
print('=' * 60)
print('EXPERIMENT 3/9: Cluster Router')
print('=' * 60)
!python run_cluster.py

In [ ]:
# ── Cell 9: Dual-encoder router ───────────────────────────────────────────────
# Trains RouterMLP (if checkpoint not found) then runs step-level + prefetch/refresh.
# Expected runtime: ~35 min (5 min if artefacts loaded)
print('=' * 60)
print('EXPERIMENT 4/9: Dual-Encoder Router')
print('=' * 60)
!python run_dual_encoder.py

In [ ]:
# ── Cell 10: Attention graph router ──────────────────────────────────────────
# Attention-weighted token neighbourhood graph.
# Expected runtime: ~20 min
print('=' * 60)
print('EXPERIMENT 5/9: Attention Graph Router')
print('=' * 60)
!python run_attention.py

In [ ]:
# ── Cell 11: MLP transition graph router ─────────────────────────────────────
# MLP up/down projection co-activation graph.
# Expected runtime: ~15 min (2 min if artefacts loaded)
print('=' * 60)
print('EXPERIMENT 6/9: MLP Transition Graph Router')
print('=' * 60)
!python run_graph.py

In [ ]:
# ── Cell 12: Speculative decoding (chain draft) ───────────────────────────────
# Standard sequential chain-based speculative decoding baseline.
# Expected runtime: ~20 min
print('=' * 60)
print('EXPERIMENT 7/9: Speculative Decoding (chain draft)')
print('=' * 60)
!python run_speculative.py

In [ ]:
# ── Cell 13: Retrieval-based speculative decoding ─────────────────────────────
# Builds 50k-completion index then runs retrieval speculative decoding.
# NOTE: This is a negative result (acceptance rate 0.2–1.1%) but included for completeness.
# Expected runtime: ~60 min (index build ~40 min + eval ~20 min)
print('=' * 60)
print('EXPERIMENT 8/9: Retrieval Speculative Decoding')
print('=' * 60)
!python run_speculative_retrieval.py

In [ ]:
# ── Cell 14: Plots ────────────────────────────────────────────────────────────
# Generates all summary figures from the CSV results.
# Expected runtime: ~1 min
print('=' * 60)
print('EXPERIMENT 9/9: Generate Plots')
print('=' * 60)
!python run_plots.py

In [ ]:
# ── Cell 15: Summary of results ───────────────────────────────────────────────
import pandas as pd
import os

print('Results directory contents:')
for f in sorted(os.listdir('results')):
    size = os.path.getsize(f'results/{f}')
    print(f'  {f:<50s} {size/1024:>8.1f} KB')

# Print the consolidated results table if it exists
all_csv = 'results/all_results.csv'
if os.path.exists(all_csv):
    print('\n=== All Results ===')
    df = pd.read_csv(all_csv)
    with pd.option_context('display.max_rows', 50, 'display.max_columns', 20,
                           'display.width', 120, 'display.float_format', '{:.3f}'.format):
        print(df.to_string(index=False))

In [ ]:
# ── Cell 16: Package results for download ────────────────────────────────────
# Zips the entire results/ directory → /kaggle/working/results.zip
# Download it from the Output tab on the right side of this notebook page.
import shutil, os

OUTPUT_ZIP = '/kaggle/working/results'
shutil.make_archive(OUTPUT_ZIP, 'zip', 'results')

zip_size = os.path.getsize(OUTPUT_ZIP + '.zip') / 1e6
print(f'results.zip created: {zip_size:.1f} MB')
print()
print('To download:')
print('  1. Click the Output tab (right panel of this notebook)')
print('  2. Find results.zip and click the download icon')
print()
print('Locally, unzip into your dsts/results/ folder:')
print('  cd /path/to/dsts')
print('  unzip ~/Downloads/results.zip -d results/')
print('  python run_plots.py   # regenerate plots locally if needed')